In [10]:
import numpy as np
import matplotlib.pyplot as plt

# Roboterparameter
L1 = 1.0  # Länge des ersten Arms
L2 = 1.0  # Länge des zweiten Arms

# Zielposition
x_target = 1.0
y_target = 1.0

# Vorwärtskinematik
def forward_kinematics(theta):
    theta1, theta2 = theta
    x = L1 * np.cos(theta1) + L2 * np.cos(theta1 + theta2)
    y = L1 * np.sin(theta1) + L2 * np.sin(theta1 + theta2)
    return np.array([x, y])

# Jacobi-Matrix berechnen
def jacobian(theta):
    theta1, theta2 = theta
    j11 = -L1 * np.sin(theta1) - L2 * np.sin(theta1 + theta2)
    j12 = -L2 * np.sin(theta1 + theta2)
    j21 =  L1 * np.cos(theta1) + L2 * np.cos(theta1 + theta2)
    j22 =  L2 * np.cos(theta1 + theta2)
    return np.array([[j11, j12],
                     [j21, j22]])

# Inverse Kinematik mit Jacobi-Methode
def inverse_kinematics(x_goal, y_goal, theta_init=[0.1, 0.1], max_iters=100, tol=1e-4):
    theta = np.array(theta_init)
    for i in range(max_iters):
        pos = forward_kinematics(theta)
        error = np.array([x_goal, y_goal]) - pos
        if np.linalg.norm(error) < tol:
            break
        J = jacobian(theta)
        # Pseudoinverse der Jacobi-Matrix
        J_pinv = np.linalg.pinv(J)
        delta_theta = J_pinv @ error
        theta += delta_theta
    return theta, i+1

# Ausführen
theta_solution, iterations = inverse_kinematics(x_target, y_target)
x_sol, y_sol = forward_kinematics(theta_solution)

# Ausgabe
print(f"Lösung nach {iterations} Iterationen:")
print(f"Theta1: {np.degrees(theta_solution[0])%360:.2f}°, Theta2: {np.degrees(theta_solution[1])%360:.2f}°")
print(f"Endeffektor-Position: x = {x_sol:.3f}, y = {y_sol:.3f}")


Lösung nach 7 Iterationen:
Theta1: 0.00°, Theta2: 90.00°
Endeffektor-Position: x = 1.000, y = 1.000


In [1]:
import numpy as np

# Längen der Armsegmente
L1 = 1.0
L2 = 1.0
L3 = 0.5

# Vorwärtskinematik
def forward_kinematics(theta):
    θ1, θ2, θ3 = theta
    x = L1 * np.cos(θ1) + L2 * np.cos(θ1 + θ2) + L3 * np.cos(θ1 + θ2 + θ3)
    y = L1 * np.sin(θ1) + L2 * np.sin(θ1 + θ2) + L3 * np.sin(θ1 + θ2 + θ3)
    z = 0.5 * np.sin(θ3)  # Beispiel für Z-Komponente (vereinfachte Höhenbewegung)
    return np.array([x, y, z])

# Jacobi-Matrix (numerisch)
def jacobian_numeric(theta, delta=1e-6):
    J = np.zeros((3, 3))
    f0 = forward_kinematics(theta)
    for i in range(3):
        theta_perturbed = np.array(theta)
        theta_perturbed[i] += delta
        fi = forward_kinematics(theta_perturbed)
        J[:, i] = (fi - f0) / delta
    return J

# Inverse Kinematik mit Jacobi
def inverse_kinematics(target_pos, theta_init=[0.1, 0.1, 0.1], tol=1e-4, max_iters=100):
    theta = np.array(theta_init)
    for i in range(max_iters):
        pos = forward_kinematics(theta)
        error = target_pos - pos
        if np.linalg.norm(error) < tol:
            break
        J = jacobian_numeric(theta)
        d_theta = np.linalg.pinv(J) @ error
        theta += d_theta
    return theta, i+1

# Zielposition
target = np.array([1.5, 0.5, 0.1])

# Berechnung
theta_sol, iters = inverse_kinematics(target)

# Ergebnis
print(f"Berechnet nach {iters} Iterationen:")
print(f"Theta1 = {np.degrees(theta_sol[0]):.2f}°, Theta2 = {np.degrees(theta_sol[1]):.2f}°, Theta3 = {np.degrees(theta_sol[2]):.2f}°")
print(f"Endposition: {forward_kinematics(theta_sol)}")


Berechnet nach 14 Iterationen:
Theta1 = -767.88°, Theta2 = 820.30°, Theta3 = 11.54°
Endposition: [1.49999123 0.50000954 0.1       ]


In [ ]:
import sympy as sp

# Anzahl DOF
n = 3

# Symbolische Gelenkwinkel q0, q1, ..., q(n-1)
q = sp.symbols(f'q0:{n}')

# Beispielhafte DH-Parameter für 3DOF-Roboter (alle alpha = 0)
dh_params = [
    {'a': 1.0, 'alpha': 0, 'd': 0, 'theta': q[0]},
    {'a': 1.0, 'alpha': 0, 'd': 0, 'theta': q[1]},
    {'a': 1.0, 'alpha': 0, 'd': 0, 'theta': q[2]},
]

# DH-Transformationsmatrix (Symbolisch)
def dh_transform(a, alpha, d, theta):
    return sp.Matrix([
        [sp.cos(theta), -sp.sin(theta)*sp.cos(alpha),  sp.sin(theta)*sp.sin(alpha), a*sp.cos(theta)],
        [sp.sin(theta),  sp.cos(theta)*sp.cos(alpha), -sp.cos(theta)*sp.sin(alpha), a*sp.sin(theta)],
        [0,              sp.sin(alpha),               sp.cos(alpha),               d],
        [0,              0,                           0,                           1]
    ])

# Gesamte Transformationsmatrix berechnen
T = sp.eye(4)
for param in dh_params:
    T = T * dh_transform(param['a'], param['alpha'], param['d'], param['theta'])

# Endeffektorposition (x, y, z)
position = T[:3, 3]
print("Endeffektorposition (symbolisch):")
sp.pprint(position)

# Jacobi-Matrix berechnen (∂x/∂qᵢ, ∂y/∂qᵢ, ∂z/∂qᵢ)
J = sp.Matrix.hstack(*[position.diff(joint) for joint in q])
print("\nJacobi-Matrix (symbolisch):")
sp.pprint(J)


# Optional: numerisch evaluieren
# Beispielwerte für die Gelenkwinkel in Radiant
values = {q[0]: 0.0, q[1]: sp.pi/4, q[2]: sp.pi/6}
pos_num = position.evalf(subs=values)
J_num = J.evalf(subs=values)

print("\nNumerische Endeffektorposition:")
sp.pprint(pos_num)

print("\nNumerische Jacobi-Matrix:")
sp.pprint(J_num)


Endeffektorposition (symbolisch):
⎡1.0⋅(-sin(q₀)⋅sin(q₁) + cos(q₀)⋅cos(q₁))⋅cos(q₂) + 1.0⋅(-sin(q₀)⋅cos(q₁) - si ↪
⎢                                                                              ↪
⎢1.0⋅(-sin(q₀)⋅sin(q₁) + cos(q₀)⋅cos(q₁))⋅sin(q₂) + 1.0⋅(sin(q₀)⋅cos(q₁) + sin ↪
⎢                                                                              ↪
⎣                                                                              ↪

↪ n(q₁)⋅cos(q₀))⋅sin(q₂) - 1.0⋅sin(q₀)⋅sin(q₁) + 1.0⋅cos(q₀)⋅cos(q₁) + 1.0⋅cos ↪
↪                                                                              ↪
↪ (q₁)⋅cos(q₀))⋅cos(q₂) + 1.0⋅sin(q₀)⋅cos(q₁) + 1.0⋅sin(q₀) + 1.0⋅sin(q₁)⋅cos( ↪
↪                                                                              ↪
↪  0                                                                           ↪

↪ (q₀)⎤
↪     ⎥
↪ q₀) ⎥
↪     ⎥
↪     ⎦

Jacobi-Matrix (symbolisch):
⎡1.0⋅(sin(q₀)⋅sin(q₁) - cos(q₀)⋅cos(q₁))⋅sin(q₂) + 1.0⋅(-sin(q₀)⋅cos(q₁) - sin ↪
⎢   

In [ ]:
#spectral decomposition
import numpy as np

M = np.array([
    [2, -1, 2],
    [-1, 10, -2],
    [2, -2, 5]
])

eigenvalues, q = np.linalg.eig(M)
A = np.diag(eigenvalues)
print(q @ A @ q.T)


Eigenwerte:
[11.  1.  5.]
Eigenvektoren (als Spalten):
[[-1.82574186e-01  8.94427191e-01  4.08248290e-01]
 [ 9.12870929e-01 -2.84004107e-16  4.08248290e-01]
 [-3.65148372e-01 -4.47213595e-01  8.16496581e-01]]
[[ 2. -1.  2.]
 [-1. 10. -2.]
 [ 2. -2.  5.]]


In [ ]:
#singular value decomposition
import numpy as np

M = np.array([
    [3, 2, 2],
    [2, 3, -2]
])

S_l = M @ M.T
S_r = M.T @ M

eigenvalues_S_l, U = np.linalg.eig(S_l)
eigenvalues_S_r, V = np.linalg.eig(S_r)

sort_l = np.argsort(eigenvalues_S_l)[::-1]
eigenvalues_S_l = eigenvalues_S_l[sort_l]
U = U[:,sort_l]

sort_r = np.argsort(eigenvalues_S_r)[::-1]
eigenvalues_S_r = eigenvalues_S_r[sort_r]
V = V[:,sort_r]

singular_values = [np.sqrt(x) for x in eigenvalues_S_l]
E = np.diag(singular_values)
E = np.hstack((E, np.zeros((E.shape[0], 1))))


print(U@E@V.T)

#hier ist das vorzeichen richtig...
U, S, Vt = np.linalg.svd(M)
print(U@E@Vt)


[[-7.07106781e-01 -7.07106781e-01 -4.55680392e-17]
 [ 2.35702260e-01 -2.35702260e-01  9.42809042e-01]
 [-6.66666667e-01  6.66666667e-01  3.33333333e-01]]
[[-3. -2. -2.]
 [-2. -3.  2.]]
[[ 3.  2.  2.]
 [ 2.  3. -2.]]
